# EORSSD Saliency Detection -- Training & Evaluation

Trains and compares five models on the EORSSD optical-remote-sensing saliency dataset: a plain CNN baseline, U-Net, VGG16-UNet, ResNet34-UNet, and Attention ResNet-UNet.

**Run this on a GPU runtime** (Kaggle: Settings -> Accelerator -> GPU; Colab: Runtime -> Change runtime type -> GPU). On CPU, drop `--epochs` way down (e.g. 2) just to confirm things run -- full training needs a GPU.

**Before running:** attach the EORSSD dataset as a Kaggle "Add Data" input (or, on Colab, upload/mount it) so `DATA_ROOT` below resolves correctly.

In [ ]:
import os

# If this notebook is running from inside the cloned repo (e.g. uploaded as a
# Kaggle notebook with the repo's files alongside it), `src/` will already be
# present. Otherwise, clone the repo.
if not os.path.exists("src"):
    REPO_URL = "https://github.com/jiyadesai07/saliency-detection-eorssd.git"
    os.system(f"git clone {REPO_URL} repo_tmp && cp -r repo_tmp/* . && rm -rf repo_tmp")

# Kaggle's preinstalled PyTorch sometimes targets only newer GPU architectures
# (it has dropped Pascal/sm_60 support, which breaks outright on a P100). Pin a
# known-good, broadly-compatible build explicitly instead of trusting whatever
# happens to be preinstalled on the assigned GPU.
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q -r requirements.txt

In [ ]:
import glob
from pathlib import Path

# Auto-detect a Kaggle EORSSD input dataset (folder containing train-images/).
# Falls back to a local ./EORSSD-Dataset/EORSSD if you've placed the data there
# yourself (e.g. on Colab after unzipping).
candidates = glob.glob("/kaggle/input/**/train-images", recursive=True)
if candidates:
    DATA_ROOT = str(Path(candidates[0]).parent)
else:
    DATA_ROOT = "EORSSD-Dataset/EORSSD"

assert (Path(DATA_ROOT) / "train-images").exists(), (
    f"Couldn't find train-images/ under {DATA_ROOT}. "
    "Attach the EORSSD dataset (Kaggle: Add Data) or set DATA_ROOT manually."
)
print("Using DATA_ROOT =", DATA_ROOT)
print("train images:", len(list((Path(DATA_ROOT) / "train-images").iterdir())))
print("test images:", len(list((Path(DATA_ROOT) / "test-images").iterdir())))

## Train all five models

Each call is a fresh `scripts/train.py` run -- best checkpoint (by validation loss) is saved to `checkpoints/{model}_best.pth`. Adjust `EPOCHS`/`BATCH_SIZE` to your GPU's memory and time budget; 60 epochs at 256x256 with a ResNet/VGG encoder comfortably fits a single T4/P100.

In [ ]:
MODELS = ["cnn", "unet", "vgg_unet", "resnet_unet", "attention_unet"]
EPOCHS = 60
BATCH_SIZE = 16

for model_name in MODELS:
    print(f"\n========== Training {model_name} ==========")
    !python scripts/train.py --model {model_name} --data-root "{DATA_ROOT}" \
        --epochs {EPOCHS} --batch-size {BATCH_SIZE}

## Evaluate every trained model with real SOD metrics

Pixel "accuracy" is dominated by the background class on EORSSD and is a misleading metric -- a model can score 80%+ accuracy while outputting a near-blank mask. MAE, max/mean F-measure, S-measure, and E-measure (the standard ORSI-SOD benchmark protocol) are what actually separate the models here.

In [ ]:
!python scripts/evaluate.py --all --data-root "{DATA_ROOT}" --checkpoint-dir checkpoints --output-dir outputs

In [ ]:
import pandas as pd

df = pd.read_csv("outputs/comparison.csv")
df.sort_values("S-measure", ascending=False)

## Qualitative comparison

Each panel below is [input image | ground truth | predicted heatmap | overlay] for the same held-out test images, one row per model.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(len(MODELS), 1, figsize=(14, 3.2 * len(MODELS)), squeeze=False)
for ax, model_name in zip(axes[:, 0], MODELS):
    sample_path = f"outputs/{model_name}/sample_000.png"
    ax.imshow(mpimg.imread(sample_path))
    ax.set_title(model_name)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Single-image demo

The actual end-user deliverable: feed in one raw image, get back a saliency heatmap. Swap `BEST_MODEL` / the checkpoint path for whichever model wins on S-measure above.

In [ ]:
BEST_MODEL = "vgg_unet"  # update this to whichever model wins on S-measure above
demo_image = next(iter(Path(DATA_ROOT, "test-images").iterdir()))

!python scripts/predict.py --model {BEST_MODEL} \
    --checkpoint checkpoints/{BEST_MODEL}_best.pth \
    --image "{demo_image}" --output demo_heatmap.png

plt.figure(figsize=(6, 6))
plt.imshow(mpimg.imread("demo_heatmap.png")[:, :, ::-1])
plt.axis("off")
plt.show()